## pip install transformers torch pandas

In [2]:
from transformers import pipeline
import pandas as pd

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_leve = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
model_parrudo = "nlptown/bert-base-multilingual-uncased-sentiment"
model_ruim = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [3]:
analisador_leve = pipeline(
    "sentiment-analysis", 
    model=model_leve
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


In [15]:
analisador_ruim = pipeline(
    "sentiment-analysis", 
    model=model_ruim
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3108.52it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
analisador = pipeline(
    "sentiment-analysis", 
    model=model_parrudo
) #demorou 5 min

frase = "O atendimento foi péssimo, demoraram muito!"
resultado = analisador(frase)

print(f"Texto: {frase}")
print(f"Resultado: {resultado}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1945.98it/s, Materializing param=classifier.weight]                                      


Texto: O atendimento foi péssimo, demoraram muito!
Resultado: [{'label': '1 star', 'score': 0.659956693649292}]


In [29]:
frase = "pode melhorar"
resultado = analisador(frase)

print(f"Texto: {frase}")
print(f"Resultado: {resultado}")

Texto: ta um lixo, pode jogar fora
Resultado: [{'label': '1 star', 'score': 0.3061659336090088}]


In [16]:

frase = "Incrivel, gostei muito do atendimento, bem embalado"
resultado = analisador_ruim(frase)

print(f"Texto: {frase}")
print(f"Resultado: {resultado}")


Texto: Incrivel, gostei muito do atendimento, bem embalado
Resultado: [{'label': 'neutral', 'score': 0.7609632611274719}]


In [20]:
reviews = pd.read_csv('review.csv')

def classificar_sentimento(texto, model=analisador):
    resultado = analisador(texto[:512])[0] 
    return resultado['label'], resultado['score']

def categorizar(label):
    if '1' in label or '2' in label:
        return 'CRÍTICO'
    elif '3' in label:
        return 'NEUTRO'
    else:
        return 'PROMOTOR'


In [23]:
reviews[['estrelas_ia', 'confianca']] = reviews['texto']\
    .apply(
        lambda x: pd.Series(classificar_sentimento(x))
    )

reviews['categoria'] = reviews['estrelas_ia']\
    .apply(categorizar)

reviews

,id_cliente,texto,estrelas_ia,confianca,categoria
0,1,"Adorei o produto, chegou super rápido!",5 stars,0.791828,PROMOTOR
1,2,"Não comprem, é uma fraude. Veio quebrado.",1 star,0.942215,CRÍTICO
2,3,"O produto é ok, cumpre o que promete, mas é caro.",3 stars,0.631321,NEUTRO
3,4,"Suporte horrível, nunca mais volto.",1 star,0.766815,CRÍTICO
4,5,"Excelente qualidade, recomendo a todos.",5 stars,0.839149,PROMOTOR
